# Databricks AutoML para Regresión
## Predicción de Precios de Propiedades

### 🎯 Objetivo
Demostrar **Databricks AutoML** en un problema de **regresión**: predecir precio de propiedades.

### 📊 Workflow
1. Generar dataset sintético de propiedades
2. Ejecutar Databricks AutoML para regresión
3. Comparar con modelo manual
4. Analizar notebook generado
5. Evaluar métricas (RMSE, MAE, R²)

In [0]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 1000

df_properties = pd.DataFrame({
    'sqft': np.random.uniform(500, 3000, n),
    'bedrooms': np.random.randint(1, 6, n),
    'bathrooms': np.random.randint(1, 4, n),
    'age_years': np.random.randint(0, 50, n),
    'location_score': np.random.uniform(1, 10, n),
    'has_garage': np.random.choice([0, 1], n),
    'has_pool': np.random.choice([0, 1], n, p=[0.7, 0.3])
})

df_properties['price'] = (
    50000 + 
    100 * df_properties['sqft'] +
    20000 * df_properties['bedrooms'] +
    15000 * df_properties['bathrooms'] +
    (-500 * df_properties['age_years']) +
    10000 * df_properties['location_score'] +
    25000 * df_properties['has_garage'] +
    30000 * df_properties['has_pool'] +
    np.random.normal(0, 20000, n)
)

print("\n┌──────────────────────────────────────────────────────┐")
print("│       🏠 DATASET DE PROPIEDADES CREADO             │")
print("└──────────────────────────────────────────────────────┘")
print(f"\n📊 Total registros: {len(df_properties):,}")
print(f"📈 Precio promedio: ${df_properties['price'].mean():,.0f}")
print(f"📉 Precio mínimo: ${df_properties['price'].min():,.0f}")
print(f"📈 Precio máximo: ${df_properties['price'].max():,.0f}")
print(f"\n📋 Columnas ({len(df_properties.columns)}): {', '.join(df_properties.columns)}")
print("\n✅ Dataset listo para AutoML\n")
display(df_properties.head(10))

## 🤖 Ejecutar Databricks AutoML

### Parámetros Clave

```python
automl.regress(
    dataset=spark_df,
    target_col="price",
    primary_metric="rmse",
    timeout_minutes=15,
    max_trials=20
)
```

### Métricas Disponibles para Regresión

* `rmse` ⭐ (recomendado, en mismas unidades que target)
* `mse`
* `mae`
* `r2`

### Modelos que AutoML Probará

* Linear Regression (Ridge, Lasso, ElasticNet)
* Decision Tree Regressor
* Random Forest Regressor
* **XGBoost Regressor** ⭐
* **LightGBM Regressor** ⭐ (usualmente el mejor)

---

**⚠️ Nota:** El siguiente código muestra la estructura. Para ejecutar realmente, descomenta el código de AutoML.

In [0]:
# Convertir a Spark DataFrame
spark_df = spark.createDataFrame(df_properties)

print("┌──────────────────────────────────────────────────┐")
print("│        DATABRICKS AUTOML - REGRESIÓN           │")
print("└──────────────────────────────────────────────────┘")
print("\n🔍 Problema: Regresión (Predicción de Precio)")
print("⏱️  Tiempo estimado: 10-15 minutos")
print("🎯 Métrica objetivo: RMSE")
print("🤖 Modelos a probar: Linear, Decision Tree, Random Forest, XGBoost, LightGBM")

print("\n\n📝 CÓDIGO PARA EJECUTAR AUTOML:\n")
print("""from databricks import automl
import mlflow

summary = automl.regress(
    dataset=spark_df,
    target_col='price',
    primary_metric='rmse',
    timeout_minutes=15,
    max_trials=20
)

print(f"✅ Mejor modelo: {summary.best_trial.model_description}")
print(f"📊 RMSE: ${summary.best_trial.metrics['val_rmse']:,.0f}")
print(f"📊 MAE: ${summary.best_trial.metrics['val_mae']:,.0f}")
print(f"📊 R²: {summary.best_trial.metrics['val_r2_score']:.4f}")
""")

print("\n🔑 AutoML probará automáticamente:")
print("  • Linear Regression")
print("  • Decision Tree")
print("  • Random Forest")
print("  • XGBoost / LightGBM")
print("  • Hyperparameter tuning")

In [0]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("┌──────────────────────────────────────────────────┐")
print("│      ENTRENAR MODELOS MANUALES (BASELINE)      │")
print("└──────────────────────────────────────────────────┘")

X = df_properties.drop('price', axis=1)
y = df_properties['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = []
for name, model in models.items():
    print(f"\n🌳 Entrenando {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        'Modelo': name,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2
    })
    print(f"  ✅ RMSE: ${rmse:,.0f}")
    print(f"  ✅ MAE: ${mae:,.0f}")
    print(f"  ✅ R²: {r2:.4f}")

results_df = pd.DataFrame(results)

print("\n\n┌──────────────────────────────────────────────────┐")
print("│         📊 RESULTADOS DEL MODELO MANUAL           │")
print("└──────────────────────────────────────────────────┘")
print("\n" + results_df.to_string(index=False))

if results_df['R²'].max() > 0.9:
    print("\n✅ Excelente ajuste (R² > 0.90)")
elif results_df['R²'].max() > 0.7:
    print("\n👍 Buen ajuste (R² > 0.70)")
else:
    print("\n⚠️  Ajuste moderado (R² < 0.70)")

## 📝 Notebook Generado por AutoML

### Lo que AutoML genera automáticamente:

1. **Data Exploration**: Estadísticas, correlaciones, outliers
2. **Feature Engineering**: One-hot encoding, scaling
3. **Model Training**: Múltiples algoritmos + tuning
4. **Evaluation**: Métricas, feature importance, residual plots
5. **MLflow Registry**: Modelo guardado y listo para deployment

### Ventajas:
* ✅ Código reproducible
* ✅ Trazabilidad completa
* ✅ Hyperparameters documentados
* ✅ Fácil de modificar

In [0]:
import plotly.express as px
import plotly.io as pio

pio.templates.default = 'gridon'

# Simular feature importance (en AutoML real viene del mejor modelo)
feature_importance = pd.DataFrame({
    'Feature': ['sqft', 'location_score', 'bedrooms', 'bathrooms', 'has_pool', 'has_garage', 'age_years'],
    'Importance': [0.45, 0.22, 0.12, 0.09, 0.06, 0.04, 0.02]
}).sort_values('Importance', ascending=True)

print("┌──────────────────────────────────────────────────┐")
print("│         📊 FEATURE IMPORTANCE                  │")
print("└──────────────────────────────────────────────────┘")

fig = px.bar(
    feature_importance,
    x='Importance',
    y='Feature',
    orientation='h',
    title='Importancia de Features en la Predicción de Precio',
    labels={'Importance': 'Importancia', 'Feature': 'Variable'},
    text='Importance',
    template='gridon',
    color='Importance',
    color_continuous_scale='Blues'
)

fig.update_traces(
    texttemplate='%{text:.1%}',
    textposition='outside'
)

fig.update_layout(
    height=400,
    showlegend=False,
    xaxis_tickformat='.0%',
    xaxis_title='Importancia Relativa'
)

fig.show()

print("\n🔑 Insights:")
print(f"  • {feature_importance.iloc[-1]['Feature']}: Factor más importante ({feature_importance.iloc[-1]['Importance']:.0%})")
print(f"  • Top 3 features explican {feature_importance.iloc[-3:]['Importance'].sum():.0%} de la varianza")
print(f"  • {feature_importance.iloc[0]['Feature']}: Factor menos relevante ({feature_importance.iloc[0]['Importance']:.1%})")

## ⚖️ AutoML vs Manual

| Aspecto | AutoML | Manual |
|---------|--------|--------|
| **Tiempo** | 15-30 min | Horas/Días |
| **Modelos** | 10-20+ | 2-3 |
| **Tuning** | Automático | Manual |
| **MLflow** | Automático | Manual |
| **Feature Eng** | Básico | Custom |
| **Interpretabilidad** | Completa | Completa |

### ✅ Usar AutoML cuando:
* Prototipado rápido
* Baseline en minutos
* Equipos sin expertos ML
* Time-to-market crítico
* Dataset tabular estándar

### ⚠️ Usar manual cuando:
* Lógica de negocio compleja
* Custom features específicos
* Máximo performance crítico
* Arquitecturas especializadas
* Deep learning / Computer Vision

---

### 🏆 Mejor Práctica

**Workflow recomendado:**

1. 🤖 **Comenzar con AutoML** → Baseline rápido en 15 min
2. 🔍 **Analizar notebook generado** → Ver qué funcionó
3. 🔧 **Iterar manualmente** → Custom features si es necesario
4. 🚀 **Desplegar mejor modelo** → MLflow Registry

## 📝 Conclusiones

### Key Takeaways

1. **AutoML acelera desarrollo**: Baseline en minutos, 10-20 modelos probados automáticamente
2. **Código reproducible**: Notebook generado con todo el pipeline + MLflow tracking
3. **Métricas para regresión**: 
   - **RMSE**: Error en mismas unidades que target ($ en este caso)
   - **MAE**: Error absoluto promedio
   - **R²**: Proporción de varianza explicada (0-1)
4. **Feature importance**: Identifica features clave (sqft, location_score)
5. **Comparación justa**: AutoML vs Manual muestra trade-off tiempo/performance

---

### 🚀 Próximos Pasos

En los siguientes notebooks exploraremos:

* **Genie_Assisted_ML_Pipeline**: Asistente AI para pipelines completos
* **MLflow_Experiment_Tracking**: Tracking avanzado de experimentos
* **Model Serving**: Despliegue y serving de modelos
* **Feature Store**: Reutilización de features

---

### 💡 Reflexión Final

> **"AutoML democratiza ML, permitiendo a equipos crear modelos de alta calidad sin ser expertos. Pero el domain knowledge sigue siendo crítico para definir el problema correcto y validar resultados."**

**Balance perfecto:**
* 🤖 AutoML para velocidad y baseline
* 👨‍💻 Human expertise para features custom e interpretación
* 🔄 Iteración continua para mejora

---

## 🎉 ¡Felicidades!

Dominas **Databricks AutoML para Regresión**. Has visto:

✅ Cómo generar datasets de prueba  
✅ Ejecutar AutoML para regresión  
✅ Comparar con modelos manuales  
✅ Evaluar métricas (RMSE, MAE, R²)  
✅ Visualizar feature importance  
✅ Entender cuándo usar AutoML vs manual  

🚀 **Listo para el siguiente nivel de automatización con Genie y MLflow!**